In [6]:
%pip install graphviz ipywidgets --only-binary=pygraphviz "pygraphviz>=2.0.2"

Note: you may need to restart the kernel to use updated packages.


In [7]:
import csv

MINUTOS_TRANSBORDO = 4

grafo = {}
estaciones = {}
nombres = {}
lineas = {}
nombres_ruta = {}
vecinos_por_linea = {}

with open("metro_cdmx_aristas.csv", encoding="utf-8-sig", newline="") as archivo:
    for fila in csv.DictReader(archivo):
        linea = fila["route_id"]
        lineas[linea] = fila["route_short_name"].removeprefix("L")
        nombres_ruta[linea] = fila["route_long_name"]
        origen = (linea, fila["from_stop_id"])
        destino = (linea, fila["to_stop_id"])
        for nodo, columna in ((origen, "from_stop_name"), (destino, "to_stop_name")):
            nombre = fila[columna].strip()
            nombres[nodo] = nombre
            estaciones.setdefault(nombre, set()).add(nodo)
            grafo.setdefault(nodo, [])

        minutos = float(fila["mean_travel_time_min"])
        grafo[origen].append((destino, minutos))
        vecinos = vecinos_por_linea.setdefault(linea, {})
        vecinos.setdefault(origen, set()).add(destino)
        vecinos.setdefault(destino, set()).add(origen)

numeros_linea = list(lineas.values())
for linea, numero in lineas.items():
    if numeros_linea.count(numero) > 1:
        lineas[linea] = f"{numero} ({nombres_ruta[linea]})"

for andenes in estaciones.values():
    andenes_ordenados = sorted(andenes)
    for indice, primero in enumerate(andenes_ordenados):
        for segundo in andenes_ordenados[indice + 1:]:
            if primero[0] != segundo[0]:
                grafo[primero].append((segundo, MINUTOS_TRANSBORDO))
                grafo[segundo].append((primero, MINUTOS_TRANSBORDO))

posicion = {}
terminales = {}
for linea, vecinos in vecinos_por_linea.items():
    terminal_a, terminal_b = sorted(nodo for nodo in vecinos if len(vecinos[nodo]) == 1)
    terminales[linea] = (nombres[terminal_a], nombres[terminal_b])

    actual = terminal_a
    indice = 0
    while actual is not None:
        posicion[actual] = indice
        actual = next((nodo for nodo in vecinos[actual] if nodo not in posicion), None)
        indice += 1


def direccion(primera, siguiente):
    terminal_a, terminal_b = terminales[primera[0]]
    return terminal_b if posicion[siguiente] > posicion[primera] else terminal_a

In [8]:
def floyd_warshall(grafo):
    nodos = list(grafo)
    indices = {nodo: indice for indice, nodo in enumerate(nodos)}
    cantidad = len(nodos)
    infinito = float("inf")
    tiempos = [[infinito] * cantidad for _ in nodos]
    siguientes = [[None] * cantidad for _ in nodos]

    for i, nodo in enumerate(nodos):
        tiempos[i][i] = 0
        for vecino, minutos in grafo[nodo]:
            j = indices[vecino]
            if minutos < tiempos[i][j]:
                tiempos[i][j] = minutos
                siguientes[i][j] = j

    for k in range(cantidad):
        tiempos_desde_k = tiempos[k]
        for i in range(cantidad):
            tiempos_desde_i = tiempos[i]
            tiempo_hasta_k = tiempos_desde_i[k]
            if tiempo_hasta_k == infinito:
                continue
            siguientes_desde_i = siguientes[i]
            primer_paso = siguientes_desde_i[k]
            for j, tiempo_desde_k in enumerate(tiempos_desde_k):
                candidato = tiempo_hasta_k + tiempo_desde_k
                if candidato < tiempos_desde_i[j]:
                    tiempos_desde_i[j] = candidato
                    siguientes_desde_i[j] = primer_paso

    return nodos, indices, tiempos, siguientes


nodos, indices, tiempos, siguientes = floyd_warshall(grafo)


def ruta_minima(origen, destino):
    # Una estación puede tener un andén por cada línea que la atiende.
    salidas = [indices[nodo] for nodo in estaciones[origen]]
    llegadas = [indices[nodo] for nodo in estaciones[destino]]
    tiempo, fin, inicio = min(
        (tiempos[i][j], j, i)
        for i in salidas
        for j in llegadas
    )
    if tiempo == float("inf"):
        return None, []

    camino = [nodos[inicio]]
    while inicio != fin:
        inicio = siguientes[inicio][fin]
        camino.append(nodos[inicio])

    return tiempo, camino

In [13]:
import ipywidgets as widgets
from IPython.display import display


def itinerario(origen, destino):
    tiempo, camino = ruta_minima(origen, destino)
    if tiempo is None:
        return "No hay ruta posible."
    if origen == destino:
        return f"0,00 min · Ya estás en {origen}."

    tramos = []
    for nodo in camino:
        linea = nodo[0]
        if not tramos or linea != tramos[-1][0]:
            tramos.append((linea, []))
        tramos[-1][1].append(nodo)

    transbordos = len(tramos) - 1
    total_transbordos = transbordos * MINUTOS_TRANSBORDO
    tiempo_texto = f"{tiempo:.2f}".replace(".", ",")
    resumen = (
        f"Tiempo mínimo estimado: {tiempo_texto} min"
        f" - Transbordos: {transbordos} ({total_transbordos} min)"
    )

    pasos = []
    for indice, (linea, paradas) in enumerate(tramos):
        nombre_linea = lineas[linea]
        terminal = direccion(paradas[0], paradas[1])
        salida = nombres[paradas[0]]
        estaciones_tramo = " → ".join(nombres[nodo] for nodo in paradas)
        if indice == 0:
            instruccion = f"En {salida}, toma la linea {nombre_linea} en dirección {terminal}."
        else:
            linea_anterior = lineas[tramos[indice - 1][0]]
            instruccion = (
                f"En {salida}, baja de la línea {linea_anterior} y continúa por la línea "
                f"{nombre_linea} en dirección a {terminal} (+{MINUTOS_TRANSBORDO})."
            )
        pasos.append(f"{indice + 1}. {instruccion}\n   {estaciones_tramo}")

    return "\n\n".join([resumen, *pasos, f"Llegada: {destino}."])


opciones = sorted(estaciones)
selector_origen = widgets.Combobox(
    options=opciones, ensure_option=True, description="Origen"
)
selector_destino = widgets.Combobox(
    options=opciones, ensure_option=True, description="Destino"
)
boton = widgets.Button(description="Calcular ruta")
resultado = widgets.Textarea(
    description="Resultado", disabled=False, rows=3,
    layout=widgets.Layout(width="100%"),
    style={"text_color": "black"},

)

def calcular(_):
    origen = selector_origen.value
    destino = selector_destino.value
    if origen in estaciones and destino in estaciones:
        mensaje = itinerario(origen, destino)
    else:
        mensaje = "Selecciona una estación."

    resultado.rows = max(3, mensaje.count("\n") + 2)
    resultado.value = mensaje


boton.on_click(calcular)
display(widgets.VBox([selector_origen, selector_destino, boton, resultado]))

In [11]:
from graphviz import Digraph, escape
from IPython.display import SVG
from pygraphviz import AGraph


def ruta_graphviz(origen, destino):
    """Devuelve el texto DOT de la red con la ruta mínima en rojo."""
    for estacion in (origen, destino):
        if estacion not in estaciones:
            raise ValueError(f"Estación desconocida: {estacion!r}")

    _, camino = ruta_minima(origen, destino)
    nodos_ruta = set(camino)
    tramos_ruta = set(zip(camino, camino[1:]))
    ids = {nodo: f"n{indice}" for indice, nodo in enumerate(grafo)}
    aristas = {
        (salida, llegada) for salida, vecinos in grafo.items() for llegada, _ in vecinos
    }

    dot = Digraph("Metro", graph_attr={"layout": "neato", "overlap": "portho"})
    for nodo, identificador in ids.items():
        dot.node(identificador, escape(nombres[nodo]), color="red" if nodo in nodos_ruta else None)

    for salida, llegada in sorted(aristas):
        inverso = (llegada, salida)
        if (salida, llegada) in tramos_ruta:
            dot.edge(ids[salida], ids[llegada], color="red")
        elif inverso not in aristas:
            dot.edge(ids[salida], ids[llegada])
        elif inverso not in tramos_ruta and salida < llegada:
            dot.edge(ids[salida], ids[llegada], dir="none")

    return dot.source


boton_grafico = widgets.Button(description="Mostrar ruta")
vista_grafico = widgets.HTML(layout=widgets.Layout(width="100%"))


def mostrar_grafico(_):
    origen, destino = selector_origen.value, selector_destino.value
    if origen not in estaciones or destino not in estaciones:
        vista_grafico.value = "Selecciona una estación de origen y una de destino."
        return

    grafico = AGraph(string=ruta_graphviz(origen, destino))
    grafico.graph_attr["size"] = "10,7"
    svg = SVG(grafico.draw(format="svg", prog="neato")).data
    vista_grafico.value = svg.replace("<svg ", '<svg style="max-width:100%;height:100%" ', 1)


boton_grafico.on_click(mostrar_grafico)
display(widgets.VBox([selector_origen, selector_destino, boton_grafico, vista_grafico]))